In [1]:
# Import libraries
import numpy as np
import pandas as pd
import geopandas as gpd

import json

import leafmap.maplibregl as leafmap

from shapely.geometry import LineString

import os

Load files

In [2]:
# Load total pop
total_pop_folder = "Population Data"
pop_gdf = gpd.read_parquet(os.path.join(total_pop_folder, "swk_1km_2020_population_age_breakdown.parquet"))

# Load student pop
student_pop_gdf = gpd.read_parquet(os.path.join(total_pop_folder, "swk_1km_2020_population_stu_breakdown.parquet"))

# Load school pop
school_folder = "School Data"
school_gdf = gpd.read_parquet(os.path.join(school_folder, "swk_list_of_schools_2025.parquet"))

# Load combined routes
routes_folder = "OSRM Routes to Nearest School"
secondary_combined_routes_gdf = gpd.read_parquet(os.path.join(routes_folder, "secondary_combined_routes.parquet"))
primary_combined_routes_gdf = gpd.read_parquet(os.path.join(routes_folder, "primary_combined_routes.parquet"))

# Boundaries
boundary_folder = "Geographic Boundaries"
swk_districts_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_districts.geojson")) # District boundaries
swk_parlimen_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_parliament.geojson")) # Parlimen boundaries
swk_dun_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_dun.geojson")) # DUN boundaries

Prepare data

In [3]:
# Create total, primary and secondary population gdfs
total_pop_gdf = pop_gdf.copy()[["id","total_pop","x","y","geometry"]]
primary_pop_gdf = student_pop_gdf.copy()[["id","primary_school_students","x","y","geometry"]]
secondary_pop_gdf = student_pop_gdf.copy()[["id","secondary_school_students","x","y","geometry"]]

# Round numbders
total_pop_gdf["total_pop"] = total_pop_gdf["total_pop"].round(0).astype(int)
primary_pop_gdf["primary_school_students"] = primary_pop_gdf["primary_school_students"].round(0).astype(int)
secondary_pop_gdf["secondary_school_students"] = secondary_pop_gdf["secondary_school_students"].round(0).astype(int)

# Rename columns
total_pop_gdf = total_pop_gdf.rename(columns={"id":"pop_id","total_pop":"total_population"})
primary_pop_gdf = primary_pop_gdf.rename(columns={"id":"pop_id"})
secondary_pop_gdf = secondary_pop_gdf.rename(columns={"id":"pop_id"})

In [4]:
# Simplify geometry
tolerance = 0.00001  # ~1 metre, adjust if needed

primary_combined_routes_gdf["geometry"] = (
    primary_combined_routes_gdf["geometry"]
    .simplify(tolerance, preserve_topology=True)
)

secondary_combined_routes_gdf["geometry"] = (
    secondary_combined_routes_gdf["geometry"]
    .simplify(tolerance, preserve_topology=True)
)


In [5]:
# Clean routes data
def clean_combined_routes_gdf(combined_routes_gdf):
    # Remove columns
    combined_routes_gdf = combined_routes_gdf.drop(columns=["euclidean_km"])
    
    # Rename columns
    combined_routes_gdf = combined_routes_gdf.rename(columns={
        "combined_route": "travel_mode",
        "osrm_km": "dist_km",
        "osrm_min": "time_min"
    })
    
    # Round numbers
    combined_routes_gdf["dist_km"] = combined_routes_gdf["dist_km"].round(1)
    combined_routes_gdf["time_min"] = combined_routes_gdf["time_min"].round(0)
    return combined_routes_gdf

secondary_combined_routes_gdf = clean_combined_routes_gdf(secondary_combined_routes_gdf)
primary_combined_routes_gdf = clean_combined_routes_gdf(primary_combined_routes_gdf)

In [6]:
# Create primary and school gdf
primary_school_gdf = school_gdf.copy()[school_gdf["primary_secondary"]=="Primary"][["id","nama_sekolah","bil_murid","bil_guru","geometry"]]
secondary_school_gdf = school_gdf.copy()[school_gdf["primary_secondary"]=="Secondary"][["id","nama_sekolah","bil_murid","bil_guru","geometry"]]

# Rename columns
primary_school_gdf = primary_school_gdf.rename(columns={"id":"school_id","nama_sekolah":"school_name","bil_murid":"num_students","bil_guru":"num_teachers"})
secondary_school_gdf = secondary_school_gdf.rename(columns={"id":"school_id","nama_sekolah":"school_name","bil_murid":"num_students","bil_guru":"num_teachers"})

In [7]:
# merge population gdf with combined routes gdf
secondary_pop_routes_gdf = secondary_pop_gdf[["pop_id","secondary_school_students"]].merge(
    secondary_combined_routes_gdf,
    on="pop_id",
    how="left"
)

primary_pop_routes_gdf = primary_pop_gdf[["pop_id","primary_school_students"]].merge(
    primary_combined_routes_gdf,
    on="pop_id",
    how="left"
)

In [8]:
# Clean boundaries data

# Districts
swk_districts_gdf = swk_districts_gdf[["name","geometry"]].rename(columns={"name":"district"})

# DUNs
swk_dun_gdf = swk_dun_gdf[["dun","geometry"]]
swk_parlimen_gdf = swk_parlimen_gdf[["parlimen","geometry"]]

Output file paths

In [9]:
output_folder_maps = "Interactive Maps"

Join primary and secondary school data and analyse travel time changes grouped by primary school

In [10]:
# Join primary and secondary school data

# First rename columns
secondary_pop_routes_gdf_renamed = secondary_pop_routes_gdf.rename(columns={
    "school_id": "sec_school_id",
    "school_name": "sec_school_name",
    "school_x": "sec_school_x",
    "school_y": "sec_school_y",
    "time_min": "sec_time_min",})

primary_pop_routes_gdf_renamed = primary_pop_routes_gdf.rename(columns={
    "school_id": "pri_school_id",
    "school_name": "pri_school_name",
    "school_x": "pri_school_x",
    "school_y": "pri_school_y",
    "time_min": "pri_time_min",})

# Filter columns
secondary_pop_routes_gdf_renamed_filtered = secondary_pop_routes_gdf_renamed[[
    "pop_id",
    "pop_x",
    "pop_y",
    "secondary_school_students",
    "sec_school_id",
    "sec_school_name",
    "sec_school_x",
    "sec_school_y",
    "sec_time_min"]]

primary_pop_routes_gdf_renamed_filtered = primary_pop_routes_gdf_renamed[[
    "pop_id",
    "pop_x",
    "pop_y",
    "primary_school_students",
    "pri_school_id",
    "pri_school_name",
    "pri_school_x",
    "pri_school_y",
    "pri_time_min"]]

# Merge columns
merged_pop_routes_gdf = primary_pop_routes_gdf_renamed_filtered.merge(
    secondary_pop_routes_gdf_renamed_filtered,
    on=["pop_id","pop_x","pop_y"],
    how="outer"
)

# Reorder columns
merged_pop_routes_gdf = merged_pop_routes_gdf[[
    "pop_id",
    "pop_x",
    "pop_y",
    "primary_school_students",
    "secondary_school_students",
    "pri_time_min",
    "pri_school_id",
    "pri_school_name",
    "pri_school_x",
    "pri_school_y",
    "sec_time_min",
    "sec_school_id",
    "sec_school_name",
    "sec_school_x",
    "sec_school_y"]]

Analyse primary schools

In [11]:
# For each primary school, calculate the % of students attending secondary schools with travel time > 60 mins

df = merged_pop_routes_gdf.copy()

# ---- Clean + ensure numeric ----
df = df.dropna(subset=["pri_school_id", "pri_school_name", "primary_school_students"])
df["primary_school_students"] = pd.to_numeric(df["primary_school_students"], errors="coerce")
df["pri_time_min"] = pd.to_numeric(df["pri_time_min"], errors="coerce")

df = df.dropna(subset=["primary_school_students", "pri_time_min"])
df = df[df["primary_school_students"] > 0]

# For each primary school, find the % of students that are > 60 mins to secondary school
def calculate_pct_over_60_mins(group):
    total_students = group["primary_school_students"].sum()
    if total_students == 0:
        return 0
    students_over_60_mins = group[group["sec_time_min"] > 60]["primary_school_students"].sum()
    pct_over_60_mins = (students_over_60_mins / total_students) * 100
    return pct_over_60_mins

pct_over_60_mins_series = df.groupby("pri_school_id").apply(calculate_pct_over_60_mins)
pct_over_60_mins_df = pct_over_60_mins_series.reset_index()

# Merge with primary school gdf to get geometry
pct_over_60_mins_df = pct_over_60_mins_df.rename(columns={0: "pct_sec_school_over_60_mins"})
pct_over_60_mins_df = pct_over_60_mins_df.merge(
    primary_school_gdf[["school_id", "geometry", "school_name"]],
    left_on="pri_school_id",
    right_on="school_id",
    how="left"
).drop(columns=["school_id"])

# Rearrange columns
pct_over_60_mins_df = pct_over_60_mins_df[[
    "pri_school_id",
    "school_name",
    "pct_sec_school_over_60_mins",
    "geometry"]]

In [12]:
pct_over_60_mins_df[pct_over_60_mins_df["pct_sec_school_over_60_mins"] > 0]

,pri_school_id,school_name,pct_sec_school_over_60_mins,geometry
5,6.0,SEKOLAH KEBANGSAAN BATU BUNGAN,100.000000,POINT (114.82 4.0532)
6,7.0,SEKOLAH KEBANGSAAN BENAWA,100.000000,POINT (114.34 4.0317)
9,11.0,SEKOLAH KEBANGSAAN KUALA TUTOH,87.755102,POINT (114.39 4.0203)
10,12.0,SEKOLAH KEBANGSAAN LIO MATO,100.000000,POINT (115.22 3.1692)
11,13.0,SEKOLAH KEBANGSAAN LONG ANAP,100.000000,POINT (114.82 3.0635)
...,...,...,...,...
1225,1260.0,SEKOLAH KEBANGSAAN SG GENAAN,100.000000,POINT (113.38 3.0245)
1226,1261.0,SEKOLAH KEBANGSAAN SG SEBUNGAN,51.041667,POINT (113.36 3.2007)
1228,1263.0,SEKOLAH KEBANGSAAN SG. SENGIAN LABANG,38.372093,POINT (113.51 3.3773)
1230,1265.0,SEKOLAH KEBANGSAAN ULU KAKUS,84.328358,POINT (113.62 2.6519)


In [13]:
#Create categories of risk

def classify_risk(p):
    if p == 0:
        return "Low risk"
    elif p <= 30:
        return "Medium risk"
    else:
        return "High risk"

pct_over_60_mins_df["category"] = pct_over_60_mins_df["pct_sec_school_over_60_mins"].apply(classify_risk)

Create student flow network

In [14]:
# Base df
df = merged_pop_routes_gdf.copy()

# --- Aggregate students flowing from each primary to each secondary
flows = (
    df.groupby(
        ["pri_school_id", "pri_school_name", "sec_school_id", "sec_school_name"],
        dropna=False
    )["primary_school_students"]
    .sum()
    .reset_index()
    .rename(columns={"primary_school_students": "students"})
)

In [15]:
# Build gdf of school points

pri_pts = primary_school_gdf[["school_id", "school_name", "geometry"]].rename(columns={"school_id":"pri_school_id","school_name":"pri_school_name"})
sec_pts = secondary_school_gdf[["school_id", "school_name", "geometry"]].rename(columns={"school_id":"sec_school_id","school_name":"sec_school_name"})

In [16]:
# Build flow geometries

# Join flow table to get geometries for both ends
flows_geo = (
    flows
    .merge(pri_pts.rename(columns={"geometry": "pri_geom"}), on=["pri_school_id", "pri_school_name"], how="left")
    .merge(sec_pts.rename(columns={"geometry": "sec_geom"}), on=["sec_school_id", "sec_school_name"], how="left")
)

# Drop pairs where we couldn't find either endpoint geometry
flows_geo = flows_geo.dropna(subset=["pri_geom", "sec_geom"]).copy()

# Create LineStrings
flows_geo["geometry"] = flows_geo.apply(lambda r: LineString([r["pri_geom"], r["sec_geom"]]), axis=1)

flow_lines_gdf = gpd.GeoDataFrame(
    flows_geo.drop(columns=["pri_geom", "sec_geom"]),
    geometry="geometry",
    crs="EPSG:4326"
)


In [17]:
# Make sure students is numeric
flow_lines_gdf["students"] = pd.to_numeric(flow_lines_gdf["students"], errors="coerce").fillna(0)

# Tunable parameters
W_MIN, W_MAX = 1, 10 # px

# Log scale (handles heavy-tailed flows nicely)
s = flow_lines_gdf["students"].clip(lower=0).astype(float)
s_log = np.log1p(s)

vmin, vmax = s_log.min(), s_log.max()
if vmax == vmin:
    flow_lines_gdf["line_width"] = (W_MIN + W_MAX) / 2
else:
    flow_lines_gdf["line_width"] = W_MIN + (s_log - vmin) * (W_MAX - W_MIN) / (vmax - vmin)

In [18]:
# Add categories
flow_lines_gdf = flow_lines_gdf.merge(
    pct_over_60_mins_df[["pri_school_id", "category"]],on="pri_school_id", how="left"
).drop(columns=["pri_school_id","sec_school_id"])

In [19]:
flow_lines_gdf

,pri_school_name,sec_school_name,students,geometry,line_width,category
0,SEKOLAH JENIS KEBANGSAAN (CINA) CHUNG HUA,SEKOLAH MENENGAH KEBANGSAAN MARUDI,307,"LINESTRING (114.32 4.1796, 114.33 4.1836)",7.549535,Low risk
1,SEKOLAH JENIS KEBANGSAAN (CINA) HUA NAM,SEKOLAH MENENGAH KEBANGSAAN TELANG USAN,17,"LINESTRING (114.35 4.1294, 114.34 4.1915)",4.303711,Low risk
2,SEKOLAH JENIS KEBANGSAAN (CINA) KEE TEE,SEKOLAH MENENGAH KEBANGSAAN LONG LAMA,267,"LINESTRING (114.4 3.7624, 114.41 3.7622)",7.390528,Low risk
3,SEKOLAH JENIS KEBANGSAAN SUNGAI JAONG MARUDI,SEKOLAH MENENGAH KEBANGSAAN MARUDI,407,"LINESTRING (114.33 4.1829, 114.33 4.1836)",7.870911,Low risk
4,SEKOLAH JENIS KEBANGSAAN SUNGAI JAONG MARUDI,SEKOLAH MENENGAH KEBANGSAAN TELANG USAN,326,"LINESTRING (114.33 4.1829, 114.34 4.1915)",7.617956,Low risk
...,...,...,...,...,...,...
1476,SEKOLAH KEBANGSAAN SG. SENGIAN LABANG,SEKOLAH MENENGAH KEBANGSAAN SUBIS,36,"LINESTRING (113.51 3.3773, 113.75 3.8029)",5.127299,High risk
1477,SEKOLAH KEBANGSAAN SG. SENGIAN LABANG,SEKOLAH MENENGAH KEBANGSAAN SEBAUH,8,"LINESTRING (113.51 3.3773, 113.27 3.125)",3.511440,High risk
1478,SEKOLAH KEBANGSAAN TATAU,SEKOLAH MENENGAH KEBANGSAAN KEMENA,12,"LINESTRING (112.85 2.8772, 113.05 3.1469)",3.931751,Low risk
1479,SEKOLAH KEBANGSAAN TATAU,SEKOLAH MENENGAH KEBANGSAAN TATAU,1289,"LINESTRING (112.85 2.8772, 112.86 2.8857)",9.186659,Low risk


Index school data with the risk categories

In [20]:
primary_school_gdf = primary_school_gdf.rename(columns={"school_id": "pri_school_id"}).merge(
    pct_over_60_mins_df[["pri_school_id", "category"]],on="pri_school_id", how="left"
)

Index population data with new school categories

In [21]:
# Merge
primary_pop_routes_gdf = primary_pop_routes_gdf.rename(columns={"school_id": "pri_school_id"}).merge(
    pct_over_60_mins_df[["pri_school_id", "category"]],on="pri_school_id", how="left"
)

# Drop original geometry column
primary_pop_routes_gdf.drop(columns=["geometry"], inplace=True)

# Make pop_x and pop_y the geometry
primary_pop_routes_gdf = gpd.GeoDataFrame(
    primary_pop_routes_gdf,
    geometry=gpd.points_from_xy(
        primary_pop_routes_gdf["pop_x"],
        primary_pop_routes_gdf["pop_y"]
    ),
    crs="EPSG:4326"  # lon/lat
)

In [22]:
merged_pop_routes_gdf_f = merged_pop_routes_gdf[["pop_id","pop_x","pop_y","primary_school_students","secondary_school_students","pri_time_min","sec_time_min","pri_school_name","sec_school_name"]]

# Add categories
merged_pop_routes_gdf_f = merged_pop_routes_gdf_f.merge(
    pct_over_60_mins_df[["school_name", "category"]], 
    left_on="pri_school_name",
    right_on="school_name",
    how="left").drop(columns=["school_name"])

# Create points
merged_pop_routes_gdf_f["geometry"] = gpd.points_from_xy(
    merged_pop_routes_gdf_f["pop_x"],
    merged_pop_routes_gdf_f["pop_y"],
    crs="EPSG:4326"
)

# Convert to GeoDataFrame
merged_pop_routes_gdf_f = gpd.GeoDataFrame(
    merged_pop_routes_gdf_f,
    geometry="geometry",
    crs="EPSG:4326"
)

Create interactive map

In [23]:
# Build map of student flows between primary and secondary schools

# -----------------------------
# 0. Define output folders
# -----------------------------

base_output_folder = output_folder_maps   # your original output folder
assets_folder = os.path.join(base_output_folder, "assets")
os.makedirs(assets_folder, exist_ok=True)

# HTML output file
output_path = os.path.join(
    base_output_folder,
    "swk_pri_sec_student_flow_interactive_map.html"
)

# -----------------------------
# 1. Create gdfs for flow layers
# -----------------------------

# Define helper function to split gdfs to 3 categories
def split_gdf_by_category(gdf):
    gdf_low = gdf[gdf["category"]=="Low risk"].copy()
    gdf_medium = gdf[gdf["category"]=="Medium risk"].copy()
    gdf_high = gdf[gdf["category"]=="High risk"].copy()
    return gdf_low, gdf_medium, gdf_high

# Create flow layers
gdf_flow_low, gdf_flow_medium, gdf_flow_high = split_gdf_by_category(flow_lines_gdf)

# -----------------------------
# 2. Build the interactive map
# -----------------------------

x = (primary_combined_routes_gdf["pop_x"].max() + primary_combined_routes_gdf["pop_x"].min()) / 2
y = (primary_combined_routes_gdf["pop_y"].max() + primary_combined_routes_gdf["pop_y"].min()) / 2

#m = leafmap.Map(height="600px", center=[x, y], use_message_queue=True, style="street")
m = leafmap.Map(height="600px", use_message_queue=True, style="street")

# Basemap
m.add_basemap("Esri.WorldImagery", before_id=m.first_symbol_layer_id, visible=False)
m.add_tile_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Esri.WorldImagery (Basemap)",
    attribution="Esri World Imagery",
    visible=True
)

# -----------------------------
# 3. Create Flow Layers
# -----------------------------

m.add_gdf(
    gdf=gdf_flow_low,
    name="Student Flows (Low Risk)",
    layer_type="line",
    paint={"line-color": "#000000", "line-width": ["get", "line_width"]},  # very light blue
    visible=True
)

m.add_gdf(
    gdf=gdf_flow_medium,
    name="Student Flows (Medium Risk)",
    layer_type="line",
    paint={"line-color": "#6baed6", "line-width": ["get", "line_width"]},  # mid blue
    visible=True
)

m.add_gdf(
    gdf=gdf_flow_high,
    name="Student Flows (High Risk)",
    layer_type="line",
    paint={"line-color": "#2171b5", "line-width": ["get", "line_width"]},  # dark blue
    visible=True
)

# -----------------------------
# 4. Adminstrative Boundaries
# -----------------------------

# Adminstrative boundaries
m.add_gdf(
    gdf=swk_districts_gdf,
    name="District Boundaries",
    layer_type="line",
    paint={"line-color": "#ffffffeb"},
    visible=True
)

m.add_gdf(
    gdf=swk_parlimen_gdf,
    name="Parlimen Boundaries",
    layer_type="line",
    paint={"line-color": "#ffffffeb"},
    visible=False
)

m.add_gdf(
    gdf=swk_dun_gdf,
    name="DUN Boundaries",
    layer_type="line",
    paint={"line-color": "#ffffffeb"},
    visible=False
)

# -----------------------------
# 5. Add school layers
# -----------------------------

# -- Primary Schools ---
image_pri = "Interactive Maps/assets/icons/college_blue.png"

# Load normal school data
primary_geojson = json.loads(primary_school_gdf.to_json())
source_pri_other = {"type": "geojson", "data": primary_geojson}

# --- Primary Schools ---
m.add_image("pri_school_other", image_pri)
m.add_source("point_pri_other", {"type": "geojson", "data": json.loads(primary_school_gdf.to_json())})
m.add_popup("Primary Schools")
m.add_layer({
    "id": "Primary Schools",
    "type": "symbol",
    "source": "point_pri_other",
    "layout": {"icon-image": "pri_school_other", "icon-size": 0.08}
})

# --- Secondary Schools ---
image_sec = "Interactive Maps/assets/icons/college_orange.png"

secondary_geojson = json.loads(secondary_school_gdf.to_json())
source_sec = {"type": "geojson", "data": secondary_geojson}

layer_sec = {
    "id": "Secondary Schools",
    "type": "symbol",
    "source": "point_sec",
    "layout": {
        "icon-image": "sec_school",
        "icon-size": 0.08}
}

m.add_image("sec_school", image_sec)
m.add_source("point_sec", source_sec)
m.add_layer(layer_sec)
m.add_popup("Secondary Schools")

# -----------------------------
# 6. Add legend
# -----------------------------

colour_map = {
    "Low Risk (0%)": "#000000",   # black
    "Medium Risk (0-30%)":    "#6baed6",   # light blue
    "High Risk (30%+)":"#2171b5"    # darker blue
}

m.add_legend(
    title = (
    "Share of students from each <br>"
    "primary school travelling more <br>"
    "than 60 mins to secondary school"),
    legend_dict = colour_map,
    position = "bottom-right"
)

# -----------------------------
# 7. Add map title / sources
# -----------------------------

m.add_text(
    "Primary to secondary school flows, highlighting primary schools where students face long travel times during secondary school transition",
    position="top-left",
    font_size=12
)

# -----------------------------
# 8. Layer control
# -----------------------------

m.add_layer_control(
    layer_ids=[
        "Student Flows (Low Risk)",
        "Student Flows (Medium Risk)",
        "Student Flows (High Risk)",
        "Primary Schools",
        "Secondary Schools",
        "District Boundaries",
        "Parlimen Boundaries",
        "DUN Boundaries",
        "Esri.WorldImagery (Basemap)"
    ],
    position="top-left"
)

# -----------------------------
# 8. Export HTML
# -----------------------------

m.to_html(
    output=output_path,
    title="Primary to Secondary School Student Flows",
    overwrite=True
)

m


Html(children=[<leafmap.maplibregl.Map object at 0x000001C423D56B30>, Card(children=[Btn(children=[Icon(childr…